# EX — Embeddings & Semantic Search Real-World Exercises

We build a tiny semantic search engine from scratch using simple bag-of-words vectors
(no external embedding API needed), so everything runs offline. The same principles
apply directly to real embedding models.


In [ ]:
import numpy as np
import re
from collections import Counter

documents = [
    "How do I reset my password?",
    "The app crashes when I open the camera",
    "I want a refund for my last order",
    "How can I change my billing address?",
    "The mobile app is very slow on startup",
    "I forgot my account password and can't log in",
    "Can I get a refund on a cancelled subscription?",
    "My camera feature keeps freezing the app",
]

def tokenize(text):
    return re.findall(r"[a-z]+", text.lower())

vocab = sorted(set(w for d in documents for w in tokenize(d)))
vocab_index = {w: i for i, w in enumerate(vocab)}
print(len(vocab), "vocab words")


## 1. Build a Simple Embedding (Bag-of-Words Vector)
**Pointer:** real embedding models replace this counting step with a learned neural representation — the search/comparison logic afterward is identical.

In [ ]:
def embed(text):
    vec = np.zeros(len(vocab))
    for w in tokenize(text):
        if w in vocab_index:
            vec[vocab_index[w]] += 1
    return vec

doc_vectors = np.array([embed(d) for d in documents])
print(doc_vectors.shape)


## 2. Cosine Similarity Search

In [ ]:
def cosine_sim(a, b):
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    return 0.0 if denom == 0 else np.dot(a, b) / denom

def search(query, k=3):
    qvec = embed(query)
    sims = [cosine_sim(qvec, dv) for dv in doc_vectors]
    ranked = sorted(zip(documents, sims), key=lambda x: -x[1])
    return ranked[:k]

for doc, score in search("my password isn't working"):
    print(f"{score:.3f}  {doc}")


### TODO 1
Search for `'app keeps freezing'` and `'I need my money back'`. Do the top results make sense? Why does bag-of-words sometimes miss true semantic matches that a real embedding model would catch (e.g., synonyms)?

In [ ]:
# TODO: run search() for both queries and print results


<details><summary>Discussion</summary>

Bag-of-words only matches on shared *exact words*. A real embedding model would know 'refund' and 'money back' are semantically related even with zero word overlap — that's the core advantage of learned embeddings over keyword counting.
</details>

## 3. Keyword Search Baseline (TF-IDF style) vs Semantic
**Pointer:** hybrid search (keyword + semantic) usually beats either alone in production.

In [ ]:
doc_freq = Counter()
for d in documents:
    for w in set(tokenize(d)):
        doc_freq[w] += 1

def tfidf_embed(text):
    vec = np.zeros(len(vocab))
    tokens = tokenize(text)
    tf = Counter(tokens)
    for w, count in tf.items():
        if w in vocab_index:
            idf = np.log((1 + len(documents)) / (1 + doc_freq[w])) + 1
            vec[vocab_index[w]] = count * idf
    return vec

tfidf_vectors = np.array([tfidf_embed(d) for d in documents])

def search_tfidf(query, k=3):
    qvec = tfidf_embed(query)
    sims = [cosine_sim(qvec, dv) for dv in tfidf_vectors]
    return sorted(zip(documents, sims), key=lambda x: -x[1])[:k]

for doc, score in search_tfidf("billing address change"):
    print(f"{score:.3f}  {doc}")


### TODO 2
Write `hybrid_search(query, alpha=0.5, k=3)` that averages the bag-of-words cosine score and the TF-IDF cosine score for each document (`alpha` weights TF-IDF vs. bag-of-words), then returns the top k.

In [ ]:
# TODO
def hybrid_search(query, alpha=0.5, k=3):
    pass

for doc, score in hybrid_search("refund for cancelled plan"):
    print(f"{score:.3f}  {doc}")


<details><summary>Solution</summary>

```python
def hybrid_search(query, alpha=0.5, k=3):
    q_bow = embed(query)
    q_tfidf = tfidf_embed(query)
    scores = []
    for doc, bow_vec, tfidf_vec in zip(documents, doc_vectors, tfidf_vectors):
        s = alpha * cosine_sim(q_tfidf, tfidf_vec) + (1-alpha) * cosine_sim(q_bow, bow_vec)
        scores.append((doc, s))
    return sorted(scores, key=lambda x: -x[1])[:k]
```
</details>


## 4. Evaluating Search Quality — precision@k

In [ ]:
# Suppose we know the "correct" relevant doc index for a query (ground truth)
ground_truth = {
    "reset my password": 0,
    "camera app crash": 1,
    "get money back": 2,
}

def precision_at_k(query, correct_doc, k=3):
    results = [doc for doc, score in search_tfidf(query, k=k)]
    return 1.0 if documents[correct_doc] in results else 0.0

for q, correct_idx in ground_truth.items():
    print(q, "->", precision_at_k(q, correct_idx))


## Key Takeaways
- Embeddings turn text into vectors where similar meaning = close vectors (cosine similarity).
- Keyword methods (TF-IDF/BM25) still win on exact terms; hybrid search combines both strengths.
- Always evaluate search quality with a metric like precision@k against known-good answers, not just by eyeballing.
- This exact retrieval logic is the first half of a RAG pipeline (next section).
